# Gated Self-Play Reinforcement Learning — Training Loop

AlphaZero-style reinforcement-learning fine-tuning for a chess network, run on Kaggle.

Each iteration performs four steps:

1. **Self-play** — the current *best* network plays a batch of games against itself to generate training positions.
2. **Train** — a *candidate* network (initialised from best) is trained on a sliding window of recent positions.
3. **Gate** — the candidate plays a fixed opening suite against best and is promoted to the new best only if it scores at or above the gate threshold.
4. **Prune** — old generation checkpoints are thinned to save disk.

Run the cells top to bottom. All reads come from the read-only dataset mount; all writes go to `/kaggle/working`.

## 1 · Configuration

Paths, network architecture, the supervised-learning seed, and all self-play / training / gating hyperparameters. Everything the rest of the notebook depends on is defined here.

In [ ]:
import os
import glob

# ============================================================
# Kaggle paths
# ============================================================
# Read-only dataset mount that contains the inner `reinforcement_learning`
# package. If this path is wrong on a new dataset version, locate it with:
#   !find /kaggle/input -name move_lookup.json
REPO_ROOT   = "/kaggle/input/datasets/ehecatlilves/reinforcement-learning"

# Everything WRITTEN must live here — /kaggle/input is read-only.
OUTPUT_ROOT = "/kaggle/working"

# Move-index lookup table used by the Converter and the network.
LOOKUP_PATH = os.path.join(REPO_ROOT, "reinforcement_learning", "move_lookup.json")

# ============================================================
# Network architecture (must match the SL weights we seed from)
# ============================================================
NUM_RES_BLOCKS = 20
NUM_FILTERS    = 256
SE_RATIO       = 8
LEARNING_RATE  = 2e-5   # low LR for RL fine-tuning of the supervised net

# ============================================================
# Supervised-learning seed
# ============================================================
# Pick the starting network by substring match against the Models folder.
SEED_MODEL = "mediocre"   # substring match; use "updated" / "16_sl_best" later
_all = (glob.glob("/kaggle/input/models/**/*.weights.h5", recursive=True)
        or glob.glob("/kaggle/input/models/**/*.h5", recursive=True))
_match = [p for p in _all if SEED_MODEL.lower() in p.lower()]
assert _match, f"no seed matching '{SEED_MODEL}'. available: {_all}"
WEIGHT_PATH = _match[0]

# ============================================================
# Self-play
# ============================================================
SIMS_PER_MOVE           = 100    # MCTS simulations per move
GAMES_PER_ITER          = 2000   # self-play games generated per iteration
SEARCH_BATCH_SIZE       = 48     # leaves collected per network call
TEMP_THRESHOLD          = 30     # sample moves with temperature for the first N plies, then play greedily
MAX_MOVES               = 300    # hard cap on game length
RESIGN_THRESHOLD        = None   # None disables resignation
RESIGN_PLAYOUT_FRACTION = 0.1    # fraction of games that track the resign condition but play on

# ============================================================
# Training
# ============================================================
TRAIN_EPOCHS     = 1
TRAIN_BATCH_SIZE = 128
BUFFER_ITERS     = 4      # sliding window: train on positions from the last N iterations
BUFFER_SAMPLE    = 8000    # positions sampled uniformly from the window per training step

# ============================================================
# Gating — a candidate must beat best on a fixed opening suite to be promoted
# ============================================================
GATE_SIMS         = 100
GATE_SEARCH_BATCH = 48
GATE_MIN_SCORE    = 10.5   # minimum score (out of 2 * len(GATE_OPENINGS)) required to accept
GATE_OPENINGS = [
    ("1.e4 e5 2.Nf3 Nc6", ["e4", "e5", "Nf3", "Nc6"]),
    ("1.e4 e5 2.Nf3 d6",  ["e4", "e5", "Nf3", "d6"]),
    ("1.e4 e5 2.Nc3",     ["e4", "e5", "Nc3"]),
    ("1.d4 Nf6",          ["d4", "Nf6"]),
    ("1.c4 e5",           ["c4", "e5"]),
    ("1.d4 d5 2.Bf4",     ["d4", "d5", "Bf4"]),
    ("1.Nf3 Nf6 2.c4",    ["Nf3", "Nf6", "c4"]),
    ("1.e4 c5 2.Nf3",     ["e4", "c5", "Nf3"]),
    ("1.b3 e5 2.Bb2",     ["b3", "e5", "Bb2"]),
    ("1.e4 d5",           ["e4", "d5"]),
]

# ============================================================
# Checkpoint pruning
# ============================================================
PRUNE_KEEP_EVERY  = 10   # always keep every Nth generation
PRUNE_KEEP_RECENT = 3    # always keep the most recent N generations

# ============================================================
# Self-play throughput (cross-game inference batcher)
# ============================================================
USE_BATCHED_SELFPLAY = True
CONCURRENT_GAMES     = 24    # concurrent games; larger = larger GPU batches (+RAM)
SERVER_MAX_BATCH     = 256   # max leaves per GPU call

# ============================================================
# Derived output paths  (reads come from REPO_ROOT; writes go to OUTPUT_ROOT)
# ============================================================
RUN_DIR   = os.path.join(OUTPUT_ROOT, "training_runs")
CKPT_DIR  = os.path.join(RUN_DIR, "checkpoints")
LOG_DIR   = os.path.join(RUN_DIR, "logs")
LOG_FILE  = os.path.join(LOG_DIR, "training.log")

BEST_PATH = os.path.join(CKPT_DIR, "best.weights.h5")
CAND_PATH = os.path.join(CKPT_DIR, "candidate.weights.h5")
GATE_LOG  = os.path.join(LOG_DIR, "gated_iterations.csv")

for d in (CKPT_DIR, LOG_DIR):
    os.makedirs(d, exist_ok=True)

# ============================================================
# Sanity check + summary
# ============================================================
assert os.path.exists(LOOKUP_PATH), "move_lookup.json not found — check REPO_ROOT / dataset contents"

print("Repo (read):   ", REPO_ROOT)
print("Output (write):", RUN_DIR)
print("SL seed:       ", WEIGHT_PATH)
print("available seeds:", [os.path.basename(p) for p in _all])
print(f"config ready. sims/move: {SIMS_PER_MOVE} | games/iter: {GAMES_PER_ITER} "
      f"| gate: {len(GATE_OPENINGS) * 2} games, need >= {GATE_MIN_SCORE}")

## 2 · GPU check

Confirm a GPU is visible to TensorFlow before starting; self-play is far too slow on CPU.

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
print("GPUs visible to TensorFlow:", gpus)
assert gpus, "No GPU visible — set Settings > Accelerator > GPU and restart the session."

## 3 · Imports & logging

Put the dataset package on the path, import the engine components (MCTS, self-play, network, converter), and configure logging to both file and stdout.

In [ ]:
import sys
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import logging

from reinforcement_learning.monte_carlo_tree_search.mcts_v2 import MCTS, SelfPlayGame
from reinforcement_learning.helpers.converter import Converter
from reinforcement_learning.networks.big_network import BigNetwork

# force=True resets handlers so re-running this cell doesn't duplicate log lines
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.FileHandler(LOG_FILE), logging.StreamHandler()],
    force=True,
)
log = logging.getLogger("train")


print("OK")

## 4 · Engine — self-play, training, and gating

Core building blocks: network construction, self-play generation, candidate training on the replay buffer, the gate match against best, and checkpoint bookkeeping.

In [ ]:
import csv, glob, re
import numpy as np
import chess

_converter = Converter(lookup_path=LOOKUP_PATH)

def build_net():
    """Build a fresh BigNetwork with the configured architecture and a new optimizer."""
    return BigNetwork(num_res_blocks=NUM_RES_BLOCKS, num_filters=NUM_FILTERS,
                      se_ratio=SE_RATIO, learning_rate=LEARNING_RATE,
                      lookup_path=LOOKUP_PATH)

def generate_selfplay(net, n_games):
    """Play n_games of self-play with `net`, serially.

    A single shared MCTS keeps the evaluation cache warm across games.
    Returns (samples, records): training positions and per-game metadata.
    """
    mcts = MCTS(network=net, converter=_converter, num_simulations=SIMS_PER_MOVE)
    samples, records = [], []
    for gi in range(n_games):
        playout = (np.random.rand() < RESIGN_PLAYOUT_FRACTION)
        game = SelfPlayGame(
            mcts, temperature_threshold=TEMP_THRESHOLD, max_moves=MAX_MOVES,
            resign_threshold=RESIGN_THRESHOLD,
            resign_playout=(playout and RESIGN_THRESHOLD is not None),
            search_batch_size=SEARCH_BATCH_SIZE,
        )
        data = game.play()
        samples.extend(data); records.append(game.record)
        print(f"    game {gi+1}/{n_games}: {game.record['result']} "
              f"{game.record['num_moves']} moves, {len(data)} positions")
    return samples, records

def train_candidate(net, buffer_samples):
    """Train `net` on a uniform random sample from the replay buffer.

    Draws up to BUFFER_SAMPLE positions from `buffer_samples`, stacks their
    board/policy/value targets, runs one training pass, and returns a dict of
    final loss values plus the number of positions trained on.
    """
    n = len(buffer_samples); k = min(BUFFER_SAMPLE, n)
    idx = np.random.choice(n, size=k, replace=(n < BUFFER_SAMPLE))
    batch = [buffer_samples[i] for i in idx]
    bt = np.stack([s["board_tensor"] for s in batch]).astype(np.float32)
    pt = np.stack([s["policy_target"] for s in batch]).astype(np.float32)
    vt = np.array([s["value_target"] for s in batch], dtype=np.float32)
    h = net.train(bt, pt, vt, epochs=TRAIN_EPOCHS, batch_size=TRAIN_BATCH_SIZE, verbose=0)
    hh = h.history
    return {"loss": float(hh.get("loss",[0])[-1]),
            "policy_loss": float(hh.get("policy_output_loss",[0])[-1]),
            "value_loss": float(hh.get("value_output_loss",[0])[-1]), "trained_on": k}

def _play_gate_game(cand_net, best_net, opening_moves, cand_is_white):
    """Play one gate game from `opening_moves` between the candidate and best nets.

    Returns (points, result_string) where points is 1.0 / 0.5 / 0.0 from the
    candidate's perspective.
    """
    mc = MCTS(network=cand_net, converter=_converter, num_simulations=GATE_SIMS)
    mb = MCTS(network=best_net, converter=_converter, num_simulations=GATE_SIMS)
    board = chess.Board()
    for san in opening_moves:
        board.push_san(san)
    while (not board.is_game_over()
           and not SelfPlayGame._claimable_draw(board)
           and len(board.move_stack) < MAX_MOVES):
        cand_to_move = (board.turn == chess.WHITE) == cand_is_white
        m = mc if cand_to_move else mb
        root = m.advance_root(None, board)
        root = m.search_batched(root, add_noise=False, batch_size=GATE_SEARCH_BATCH)
        if not root.edges:
            break
        board.push(m.get_best_move(root, temperature=0))
    result = board.result() if board.is_game_over() else "1/2-1/2"
    if result == "1/2-1/2":
        return 0.5, result
    cand_won = ((result == "1-0") == cand_is_white)
    return (1.0 if cand_won else 0.0), result

def run_gate(cand_net, best_net):
    """Play the full gate suite (each opening as both colors) candidate vs. best.

    Returns (total_candidate_score, per_game_log).
    """
    score, log = 0.0, []
    for name, moves in GATE_OPENINGS:
        for cand_white in (True, False):
            pts, result = _play_gate_game(cand_net, best_net, moves, cand_white)
            score += pts
            log.append(f"{name} cand={'W' if cand_white else 'B'} {result} (+{pts})")
    return score, log

def _gen_index(path):
    """Extract the integer generation index from a `gen_<n>.weights.h5` filename."""
    m = re.search(r"gen_(\d+)\.weights", os.path.basename(path))
    return int(m.group(1)) if m else None

def latest_generation():
    """Return the highest generation index found in CKPT_DIR (0 if none)."""
    idx = [_gen_index(p) for p in glob.glob(os.path.join(CKPT_DIR, "gen_*.weights.h5"))]
    idx = [i for i in idx if i is not None]
    return max(idx) if idx else 0

def prune_checkpoints():
    """Thin old generation checkpoints, keeping every PRUNE_KEEP_EVERY-th generation
    and the most recent PRUNE_KEEP_RECENT.
    """
    gens = sorted(i for i in
                  (_gen_index(p) for p in glob.glob(os.path.join(CKPT_DIR, "gen_*.weights.h5")))
                  if i is not None)
    if not gens:
        return
    latest = max(gens)
    keep = {g for g in gens if g % PRUNE_KEEP_EVERY == 0 or g > latest - PRUNE_KEEP_RECENT}
    removed = 0
    for g in gens:
        if g not in keep:
            p = os.path.join(CKPT_DIR, f"gen_{g}.weights.h5")
            if os.path.exists(p):
                os.remove(p); removed += 1
    if removed:
        print(f"    pruned {removed} old generation(s); kept {sorted(keep)}")

def _log_gate_csv(row):
    """Append one iteration's gate metrics as a row in GATE_LOG."""
    write_header = not os.path.exists(GATE_LOG)
    with open(GATE_LOG, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()))
        if write_header:
            w.writeheader()
        w.writerow(row)

## 5 · Cross-game inference batcher

Optional throughput layer. Runs many self-play games on worker threads and coalesces their MCTS leaf evaluations into large GPU batches through a single inference server, so the accelerator stays busy. Enabled via `USE_BATCHED_SELFPLAY`.

In [ ]:
import threading, queue, time
import numpy as np
from concurrent.futures import ThreadPoolExecutor

class InferenceServer:
    """Single-thread network server. Game threads submit leaf tensors and block;
    the server coalesces everything waiting into one big predict_batch."""
    def __init__(self, net, max_batch=256):
        self.net = net; self.max_batch = max_batch
        self.q = queue.Queue(); self._stop = False
        self.calls = 0; self.total_leaves = 0
        self.t = threading.Thread(target=self._loop, daemon=True); self.t.start()
    def submit(self, tensors):
        """Submit leaf tensors for evaluation and block until the server responds.

        Returns (policy, value) arrays aligned with `tensors`.
        """
        box = {}; ev = threading.Event()
        self.q.put((list(tensors), box, ev))
        while not ev.wait(timeout=5.0):
            if self._stop and self.q.empty():
                raise RuntimeError("inference server stopped before responding")
        if "err" in box: raise box["err"]
        return box["policy"], box["value"]
    def _loop(self):
        """Server loop: coalesce all queued requests into one predict_batch call."""
        while not self._stop:
            try: item = self.q.get(timeout=0.1)
            except queue.Empty: continue
            batch = [item]; total = len(item[0])
            while total < self.max_batch:
                try: nxt = self.q.get_nowait()
                except queue.Empty: break
                batch.append(nxt); total += len(nxt[0])
            flat = [t for tensors, _, _ in batch for t in tensors]
            try:
                policies, values = self.net.predict_batch(flat)
                self.calls += 1; self.total_leaves += len(flat)
                off = 0
                for tensors, box, ev in batch:
                    k = len(tensors)
                    box["policy"] = policies[off:off+k]; box["value"] = values[off:off+k]
                    off += k; ev.set()
            except Exception as e:
                for tensors, box, ev in batch:
                    box["err"] = e; ev.set()
    def stop(self):
        """Signal the server thread to stop and wait for it to finish."""
        self._stop = True; self.t.join(timeout=5.0)

class NetworkProxy:
    """Looks like a BigNetwork to MCTS, but routes inference through the server."""
    def __init__(self, server): self.server = server
    def predict_batch(self, board_tensors): return self.server.submit(board_tensors)
    def predict(self, board_tensor):
        p, v = self.server.submit([board_tensor]); return p[0], v[0]

def generate_selfplay_batched(net, n_games):
    """Self-play n_games concurrently; all games' leaf evals batch together on net."""
    net.predict(np.zeros((8, 8, 20), dtype=np.float32))   # trace infer fn on main thread
    server = InferenceServer(net, max_batch=SERVER_MAX_BATCH)
    proxy = NetworkProxy(server)
    results = [None] * n_games
    def worker(i):
        conv = Converter(lookup_path=LOOKUP_PATH)
        mcts = MCTS(network=proxy, converter=conv, num_simulations=SIMS_PER_MOVE)
        playout = (np.random.rand() < RESIGN_PLAYOUT_FRACTION)
        game = SelfPlayGame(mcts, temperature_threshold=TEMP_THRESHOLD, max_moves=MAX_MOVES,
                            resign_threshold=RESIGN_THRESHOLD,
                            resign_playout=(playout and RESIGN_THRESHOLD is not None),
                            search_batch_size=SEARCH_BATCH_SIZE)
        data = game.play(); results[i] = (data, game.record)
    t0 = time.perf_counter()
    try:
        with ThreadPoolExecutor(max_workers=CONCURRENT_GAMES) as ex:
            list(ex.map(worker, range(n_games)))
    finally:
        server.stop()
    dt = time.perf_counter() - t0
    samples = [s for r in results for s in r[0]]
    records = [r[1] for r in results]
    avg = server.total_leaves / max(1, server.calls)
    print(f"    batched: {n_games} games in {dt:.0f}s | {server.calls} GPU calls, "
          f"avg batch {avg:.0f} leaves")
    return samples, records

## 6 · Run the training loop

Build the best/candidate networks, (re)seed from the SL weights if the seed changed, then run the gated loop. Remember to **Save Version** afterwards to persist the weights and logs.

In [ ]:
from collections import deque
import os, glob

def run_gated_training(num_iterations, best_net, train_net, buffer):
    """Run `num_iterations` of the gated self-play loop.

    Each iteration: generate self-play games with `best_net`, train `train_net`
    (a candidate seeded from best) on the replay `buffer`, gate the candidate
    against best, promote it to a new generation if it clears GATE_MIN_SCORE,
    then prune old checkpoints. Metrics are appended to GATE_LOG each iteration.
    """
    generation = latest_generation()
    for it in range(1, num_iterations + 1):
        t0 = time.perf_counter()
        print(f"\n===== iteration {it}/{num_iterations} (current best = gen_{generation}) =====")

        t_sp = time.perf_counter()
        gen_fn = generate_selfplay_batched if USE_BATCHED_SELFPLAY else generate_selfplay
        samples, records = gen_fn(best_net, GAMES_PER_ITER)
        sp_s = time.perf_counter() - t_sp
        buffer.append(samples)
        flat = [s for it_samples in buffer for s in it_samples]

        train_net.load(BEST_PATH)                 # candidate starts from best
        t_tr = time.perf_counter()
        loss = train_candidate(train_net, flat)
        tr_s = time.perf_counter() - t_tr
        train_net.save(CAND_PATH)

        t_g = time.perf_counter()
        cand_score, gate_log = run_gate(train_net, best_net)
        g_s = time.perf_counter() - t_g
        accepted = cand_score >= GATE_MIN_SCORE

        if accepted:
            generation += 1
            train_net.save(os.path.join(CKPT_DIR, f"gen_{generation}.weights.h5"))
            best_net.load(CAND_PATH); best_net.save(BEST_PATH)   # promote
            verdict = f"ACCEPTED -> gen_{generation}"
        else:
            verdict = "rejected (best unchanged)"

        prune_checkpoints()
        dt = time.perf_counter() - t0
        print(f"  loss {loss['loss']:.3f} (pol {loss['policy_loss']:.3f} val {loss['value_loss']:.3f}) | buffer {len(flat)} pos")
        print(f"  gate: candidate {cand_score}/{len(GATE_OPENINGS)*2}  -> {verdict}")
        print(f"  time: selfplay {sp_s:.0f}s  train {tr_s:.0f}s  gate {g_s:.0f}s  total {dt:.0f}s")
        _log_gate_csv({"iteration": it, "generation": generation, "cand_score": cand_score,
                       "accepted": int(accepted), "loss": round(loss["loss"],4),
                       "policy_loss": round(loss["policy_loss"],4), "value_loss": round(loss["value_loss"],4),
                       "buffer_positions": len(flat), "trained_on": loss["trained_on"],
                       "selfplay_s": round(sp_s,1), "train_s": round(tr_s,1), "gate_s": round(g_s,1),
                       "avg_move_s": records[0].get("avg_move_seconds") if records else None})
    print("\ndone. Commit the notebook (Save Version) to keep weights + logs.")


# ---- setup (auto-reseed if the SL seed changed) ----
best_net  = build_net()
train_net = build_net()          # fresh optimizer + new LR
train_net.load(BEST_PATH)

_marker = os.path.join(CKPT_DIR, "seed_source.txt")
_prev = open(_marker).read().strip() if os.path.exists(_marker) else None
_fresh = (not os.path.exists(BEST_PATH)) or (_prev != WEIGHT_PATH)

if _fresh:
    for p in (glob.glob(os.path.join(CKPT_DIR, "best.weights.h5"))
              + glob.glob(os.path.join(CKPT_DIR, "candidate.weights.h5"))
              + glob.glob(os.path.join(CKPT_DIR, "gen_*.weights.h5"))):
        os.remove(p); print("removed", os.path.basename(p))
    best_net.load(WEIGHT_PATH)
    best_net.save(BEST_PATH)
    with open(_marker, "w") as f:
        f.write(WEIGHT_PATH)
    print("FRESH START — seeded best from new SL:", WEIGHT_PATH)
else:
    best_net.load(BEST_PATH)
    print("resumed best from", BEST_PATH)

train_net.load(BEST_PATH)
buffer = deque(maxlen=BUFFER_ITERS)

# ---- go ----
run_gated_training(5, best_net, train_net, buffer)